# Encode train-clean-360 on Colab (v2 — robust resume)

Improvements over v1:
- Shell-based listing of already-done files (bypasses `os.listdir` I/O errors on huge Drive folders)
- Force-remount Drive at start
- In-memory `done_stems` set (no per-file Drive API calls in the hot loop)
- Atomic saves: write `.tmp` then `os.replace` — crashes can't leave half-written `.npz`
- Startup sweep of any leftover `.tmp` files from prior crashes
- Per-file size validation after write (catches truncated uploads)
- Loud warnings if any staged files fail to sync after retries

**Runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 0: GPU check + Drive force-remount + shell-based existing-file listing
import os, subprocess, time
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/features_360'
os.makedirs(DRIVE_FEATURES, exist_ok=True)


def shell_ls(path, timeout=600):
    """Run `ls` on Drive folder. Returns list of names (no path). Captures stderr so we
    can see real failure reasons instead of silently getting an empty list."""
    r = subprocess.run(
        f"ls -1 '{path}'", shell=True, capture_output=True, text=True, timeout=timeout,
    )
    if r.returncode != 0:
        raise RuntimeError(f'ls failed (rc={r.returncode}): {r.stderr.strip()[:300]}')
    return [line.strip() for line in r.stdout.splitlines() if line.strip()]


def cleanup_stale_tmps(path):
    """Delete any .tmp files left from prior crashes. Uses shell find for reliability."""
    r = subprocess.run(
        f"find '{path}' -maxdepth 1 -name '*.tmp' -delete -print",
        shell=True, capture_output=True, text=True, timeout=300,
    )
    removed = [l for l in r.stdout.splitlines() if l.strip()]
    if removed:
        print(f'Cleaned up {len(removed)} stale .tmp files from prior crashes')
    return len(removed)


def build_done_stems(names):
    """From a list of filenames, return the set of utterance IDs (stems without .npz)."""
    done = set()
    for n in names:
        if n.endswith('.npz') and n != 'norm_stats.npz':
            done.add(n[:-4])
    return done


print('Cleaning up any stale .tmp files on Drive...')
cleanup_stale_tmps(DRIVE_FEATURES)

print('\nListing existing Drive files via shell ls (30-90s for 50K+ files)...')
t0 = time.time()
names = shell_ls(DRIVE_FEATURES)
done_stems = build_done_stems(names)
print(f'Already encoded on Drive: {len(done_stems)} files (listing took {time.time()-t0:.1f}s)')

In [ ]:
# Step 1: Install SPARC
!pip install -q speech-articulatory-coding soundfile

In [ ]:
# Step 2: Download + extract LibriSpeech train-clean-360 (ephemeral — re-downloads each session)
import os, glob

if not os.path.exists('LibriSpeech/train-clean-360'):
    print('Downloading train-clean-360 (~13 GB)...')
    !wget -q --show-progress http://www.openslr.org/resources/12/train-clean-360.tar.gz
    print('Extracting...')
    !tar xzf train-clean-360.tar.gz
    !rm train-clean-360.tar.gz
else:
    print('Already downloaded.')

files = sorted(glob.glob('LibriSpeech/train-clean-360/**/*.flac', recursive=True))
print(f'train-clean-360: {len(files)} files')

In [ ]:
# Step 3: Load SPARC on GPU + speed check (use only not-yet-done files for realistic timing)
from sparc import load_model
from pathlib import Path
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading SPARC on {device}...')
coder = load_model('en', device=device)
print('SPARC loaded.')

# Pick 4 files that haven't been encoded yet for timing (1 warmup + 3 timed)
pending = [f for f in files if Path(f).stem not in done_stems]
sample = pending[:4] if len(pending) >= 4 else files[:4]

if sample:
    _ = coder.encode(sample[0])  # warm-up (result discarded)
    t0 = time.time()
    for f in sample[1:4]:
        _ = coder.encode(f)
    avg = (time.time() - t0) / max(1, len(sample) - 1)
    remaining = len(pending)
    print(f'Per-file encode time: {avg:.2f}s')
    print(f'Remaining to encode: {remaining} files')
    print(f'Estimated total time: {avg * remaining / 3600:.1f} hours')
else:
    print('All files already encoded on Drive — nothing to do.')

In [ ]:
# Step 4: Encode + batched atomic sync to Drive, with post-write size validation
import numpy as np
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

LOCAL_STAGE = '/content/staging'
os.makedirs(LOCAL_STAGE, exist_ok=True)

SYNC_EVERY   = 50     # files per Drive sync batch (smaller = more resilient to throttling)
MIN_NPZ_SIZE = 500    # bytes; any file smaller than this after upload is suspect
MAX_FLUSH_ATTEMPTS = 5

success = 0            # encoded + synced this session
already_done = len(done_stems)
failed = 0             # encode failures this session
sync_failures = 0      # individual sync attempts that errored
sync_batch = []        # stems pending upload to Drive
t_start = time.time()


def sync_one(name, max_retries=4):
    """Copy staging/<name>.npz to Drive atomically with exponential backoff on failure.
    Validates final size. Returns True on success, (False, err) on failure.
    Cleans up any stale .tmp on Drive on failure."""
    src = Path(LOCAL_STAGE) / name
    dst = Path(DRIVE_FEATURES) / name
    tmp = Path(DRIVE_FEATURES) / (name + '.tmp')
    backoff = 5
    last_err = None
    for attempt in range(max_retries):
        try:
            shutil.copy2(str(src), str(tmp))
            size = tmp.stat().st_size
            if size < MIN_NPZ_SIZE:
                try: tmp.unlink()
                except Exception: pass
                raise RuntimeError(f'uploaded size {size} bytes below threshold {MIN_NPZ_SIZE}')
            os.replace(str(tmp), str(dst))
            src.unlink()
            return True
        except Exception as e:
            last_err = str(e)
            try:
                if tmp.exists(): tmp.unlink()
            except Exception:
                pass
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)  # 5s, 10s, 20s, 40s (cap 60)
    return False, last_err


def flush_sync_batch():
    """Upload every staged file to Drive. Successes removed from sync_batch; failures remain."""
    global sync_failures
    synced_count = 0
    for name in list(sync_batch):  # iterate a copy because we mutate the list
        result = sync_one(name)
        if result is True:
            sync_batch.remove(name)
            synced_count += 1
        else:
            _, err = result
            sync_failures += 1
            if sync_failures < 10:
                print(f'  sync fail {name}: {err}')
    return synced_count


pbar = tqdm(files, desc='Encoding', initial=already_done, total=len(files))

for f in pbar:
    utt_id = Path(f).stem
    if utt_id in done_stems:
        continue

    stage_path = Path(LOCAL_STAGE) / f'{utt_id}.npz'

    try:
        code = coder.encode(f)
        ema      = np.asarray(code['ema'],      dtype=np.float32)
        pitch    = np.asarray(code['pitch'],    dtype=np.float32).squeeze(-1)
        loudness = np.asarray(code['loudness'], dtype=np.float32).squeeze(-1)
        spk_emb  = np.asarray(code['spk_emb'],  dtype=np.float32)
        m = min(ema.shape[0], pitch.shape[0], loudness.shape[0])
        ema, pitch, loudness = ema[:m], pitch[:m], loudness[:m]
        # Save directly to stage_path. Local staging is ephemeral; if an encode is
        # interrupted we get a partial file but done_stems isn't updated until after
        # this save, so the utterance is cleanly retried on next session.
        # (Atomic rename on the Drive side is handled in sync_one.)
        np.savez_compressed(str(stage_path), ema=ema, pitch=pitch, loudness=loudness, spk_emb=spk_emb)
        sync_batch.append(f'{utt_id}.npz')
        done_stems.add(utt_id)
        success += 1
    except Exception as e:
        if failed < 10:
            print(f'encode fail {utt_id}: {e}')
        failed += 1
        try:
            if stage_path.exists(): stage_path.unlink()
        except Exception:
            pass
        continue

    if len(sync_batch) >= SYNC_EVERY:
        flush_sync_batch()
        time.sleep(1)  # breathing room so Drive doesn't throttle
        elapsed = time.time() - t_start
        rate = success / max(elapsed, 1)
        remaining = len(files) - success - already_done - failed
        eta_h = remaining / max(rate, 0.01) / 3600
        pbar.set_postfix(new=success, failed=failed, eta_h=f'{eta_h:.1f}')

# --- Final flush with bounded retries ---
for attempt in range(MAX_FLUSH_ATTEMPTS):
    if not sync_batch:
        break
    print(f'Final flush attempt {attempt+1}/{MAX_FLUSH_ATTEMPTS}: {len(sync_batch)} files to Drive...')
    synced = flush_sync_batch()
    print(f'  synced {synced}, remaining stuck: {len(sync_batch)}')
    if synced == 0 and sync_batch:
        time.sleep(10)

elapsed = time.time() - t_start
print(f'\nDone in {elapsed/3600:.2f}h')
print(f'  encoded this session: {success}')
print(f'  already on Drive at start: {already_done}')
print(f'  encode failures: {failed}')
print(f'  sync attempts that errored: {sync_failures}')
print(f'  Drive now holds: {len(done_stems)} / {len(files)} files')

if sync_batch:
    print(f'\n⚠️  WARNING: {len(sync_batch)} files stuck in {LOCAL_STAGE}, NOT synced to Drive:')
    for n in sync_batch[:20]:
        print(f'    {n}')
    if len(sync_batch) > 20:
        print(f'    ... and {len(sync_batch) - 20} more')
    print('  Their encoded data is only on the ephemeral Colab VM. Re-run this cell to retry.')
    print('  If the runtime dies first, they will be re-encoded on next session.')


## After encoding completes

### 1. Download features from Google Drive

drive.google.com → `articulatory_tts/features_360/` → right-click → Download (auto-zips).

### 2. Extract on your Mac

```bash
cd ~/projects/articulatory-tts/data
mkdir -p features_360_raw
cd features_360_raw
unzip ~/Downloads/features_360-*.zip
find . -mindepth 2 -name '*.npz' -exec mv {} . \;
find . -type d -empty -delete
```

### 3. Merge & retrain as in the main pipeline.